# Aula 15 - Notebook: Simulação de Vazamentos, Injeção de Falhas e Desvio Automático

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook executamos um **teste de estresse em malha fechada** aplicando injeção de falhas e distúrbios topológicos:
1. **Detecção Algébrica de Vazamentos:** Localização de perdas de massa pelo resíduo matricial $\vec{r} = B \cdot \vec{Q} - \vec{S}$;
2. **Injeção de Obstruções e Falhas de Pistão:** Bloqueio pontual de canais de escoamento e reconfiguração topológica instantânea;
3. **Desvio Automático em Tempo Real:** Recálculo de rotas de fuga/reciclo e disparo de intertravamentos de segurança de montante.


### 2. Funções de Formatação Tabular ASCII


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


### 3. Modelo Topológico Integrado e Construtor da Malha


In [ ]:
from typing import List, Dict, Any, Tuple, Optional
import heapq

class GrafoProcesso:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float,
                           tag_valvula: str, meio_transporte: str = "Duto"):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Aresta": f"e{len(self.arestas_detalhes)+1}",
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Atuador / ISA": tag_valvula,
            "Meio": meio_transporte
        })

    def construir_matriz_incidencia(self) -> List[List[int]]:
        m = len(self.arestas_detalhes)
        B = [[0] * m for _ in range(self.n)]
        for j, a in enumerate(self.arestas_detalhes):
            u = self.v_to_idx[a["Origem"]]
            v = self.v_to_idx[a["Destino"]]
            B[u][j] = -1
            B[v][j] = 1
        return B

def criar_rede_estresse() -> GrafoProcesso:
    nos = [
        "FN-101_Recepcao", "ALIM-101_Calha", "EST-201_Entrada", "BAL-301_Pesagem",
        "VIS-401_Inspecao", "DIV-501_Decisao", "EJE-602_EstacaoB", "EJE-603_EstacaoC",
        "CALHA-602_Secundario", "CALHA-603_Rejeito", "CALHA-701_Terminal",
        "SILO-701_Aprovados", "SILO-702_Secundario", "SILO-703_Rejeito"
    ]
    g = GrafoProcesso(nos)
    g.adicionar_tubulacao("FN-101_Recepcao", "ALIM-101_Calha", 1.2, "XV-101", "Duto Vertical")
    g.adicionar_tubulacao("ALIM-101_Calha", "EST-201_Entrada", 1.8, "c_ALIM", "Calha Vibratória")
    g.adicionar_tubulacao("EST-201_Entrada", "BAL-301_Pesagem", 2.5, "c_EST/ST-201", "Correia")
    g.adicionar_tubulacao("BAL-301_Pesagem", "VIS-401_Inspecao", 3.0, "WT-301/FT-301", "Correia")
    g.adicionar_tubulacao("VIS-401_Inspecao", "DIV-501_Decisao", 1.0, "XS-401/KSA-401", "Zona Óptica")
    g.adicionar_tubulacao("DIV-501_Decisao", "EJE-602_EstacaoB", 2.0, "ZC-602/p_POS602", "Correia")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "CALHA-602_Secundario", 0.8, "FY-602/ZSH-602", "Sopro B")
    g.adicionar_tubulacao("EJE-602_EstacaoB", "EJE-603_EstacaoC", 1.5, "ST-201", "Correia")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-603_Rejeito", 0.8, "FY-603/ZSH-601", "Sopro C")
    g.adicionar_tubulacao("EJE-603_EstacaoC", "CALHA-701_Terminal", 2.2, "Descarga Inércia", "Correia")
    g.adicionar_tubulacao("CALHA-701_Terminal", "SILO-701_Aprovados", 3.5, "XV-701", "Duto A")
    g.adicionar_tubulacao("CALHA-602_Secundario", "SILO-702_Secundario", 4.0, "XV-702", "Duto B")
    g.adicionar_tubulacao("CALHA-603_Rejeito", "SILO-703_Rejeito", 3.2, "XV-703", "Duto C")
    g.adicionar_tubulacao("SILO-702_Secundario", "FN-101_Recepcao", 12.0, "XV-104", "Elevador Reciclo")
    return g

rede = criar_rede_estresse()


### 4. Implementação do Motor de Diagnóstico e Desvio: Classe `SimuladorVazamentosFalhas`


In [ ]:
class SimuladorVazamentosFalhas:
    def __init__(self, grafo: GrafoProcesso):
        self.g = grafo
        self.B = grafo.construir_matriz_incidencia()

    def diagnosticar_balanco(self, Q_medido: List[float], S_esperado: List[float], tol: float = 1.0) -> List[Dict[str, Any]]:
        """Calcula o vetor de residuo r = B * Q - S e aponta nos com vazamento ou transbordo."""
        m = len(Q_medido)
        r = [0.0] * self.g.n
        for i in range(self.g.n):
            soma_bq = sum(self.B[i][j] * Q_medido[j] for j in range(m))
            r[i] = soma_bq - S_esperado[i]

        relatorio = []
        for i, no in enumerate(self.g.vertices):
            desvio = r[i]
            if abs(desvio) <= tol:
                status = "NORMAL (Balanço OK)"
            elif desvio > tol:
                status = f"RETENÇÃO / TRANSBORDO (+{desvio:.1f} kg/h)"
            else:
                status = f"VAZAMENTO / PERDA ({desvio:.1f} kg/h)"
            
            relatorio.append({
                "Equipamento / Nó": no,
                "Resíduo r_i (kg/h)": f"{desvio:+.1f}",
                "Diagnóstico": status
            })
        return relatorio

    def dijkstra_reencaminhamento(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        dist = {v: float('inf') for v in self.g.vertices}
        pai = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]: continue
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf'):
                    v = self.g.idx_to_v[v_idx]
                    if dist[u] + peso < dist[v]:
                        dist[v] = dist[u] + peso
                        pai[v] = u
                        heapq.heappush(heap, (dist[v], v))
        
        # Reconstrói rota
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            if atual == origem: break
            atual = pai[atual]
        return dist[destino], list(reversed(caminho)) if dist[destino] < float('inf') else []


### 5. Execução do Teste de Estresse 1: Detecção e Isolamento de Vazamento Granular


In [ ]:
simulador = SimuladorVazamentosFalhas(rede)

# Regime Nominal
# Q = [e1:1200, e2:1200, e3:1200, e4:1200, e5:1200, e6:1200, e7:180, e8:1020, e9:120, e10:900, e11:900, e12:180, e13:120, e14:0]
# Injetando vazamento de 150 kg/h no trecho e3 (apenas 1050 chegam na balança)
Q_vazamento = [1200.0, 1200.0, 1050.0, 1050.0, 1050.0, 1050.0, 157.5, 892.5, 105.0, 787.5, 787.5, 157.5, 105.0, 0.0]
S_esperado = [-1200.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 900.0, 180.0, 120.0]

diag_vazamento = simulador.diagnosticar_balanco(Q_vazamento, S_esperado)
print("=== DIAGNÓSTICO MATRICIAL DE VAZAMENTO (INJEÇÃO DE PERDA EM e3) ===")
print(formatar_tabela(diag_vazamento))


### 6. Execução do Teste de Estresse 2: Falha Crítica de Silo e Desvio Automático em Tempo Real


In [ ]:
# Cenário: Transbordo no Silo A (XV-701 bloqueada)
print(">>> SIMULANDO FALHA: BLOQUEIO TOTAL DA ENTRADA DO SILO A (p_NC701 = 1)")
rede.adj_pesos[rede.v_to_idx["CALHA-701_Terminal"]][rede.v_to_idx["SILO-701_Aprovados"]] = float('inf')

dist_a, rota_a = simulador.dijkstra_reencaminhamento("FN-101_Recepcao", "SILO-701_Aprovados")
print(f"Status da Rota para SILO-701: {'ATIVO' if dist_a < float('inf') else 'BLOQUEADO / ILHADO'}")

# Avaliando rota de contingencia para o Silo B (Secundário)
dist_b, rota_b = simulador.dijkstra_reencaminhamento("FN-101_Recepcao", "SILO-702_Secundario")
print(f"\nAcionando Rota de Contingência para SILO-702:")
print(f"-> Distância Mínima: {dist_b:.1f} m")
print(f"-> Trajetória: {' -> '.join(rota_b)}")
print("-> Ação do CLP: Eletroválvula FY-602 comutada para desvio total. Alimentador mantido em segurança!")
